#E-Commerce Customer Order Analysis

## Objective
Clean and analyze e-commerce transaction data to uncover key business
insights about the customers, products and revenue.

##Tools Used
- Python (Pandas, Numpy )
- SQL (sqlite3)

##Dataset
E-Commerce transaction dataset from kaggle

In [1]:
import pandas as pd
import numpy as np
import sqlite3

#Step - 1: Load the dataset

In [2]:
df = pd.read_csv("ecommerce.csv", encoding = "latin-1")
print("Shape: ", df.shape)
print("\nColumns: ", df.columns.tolist())
print("\n Data Types:\n", df.dtypes)
print("\nFirst 5 Rows: ")
df.head()

Shape:  (213216, 8)

Columns:  ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

 Data Types:
 InvoiceNo       object
StockCode       object
Description     object
Quantity       float64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

First 5 Rows: 


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6.0,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,12/1/2010 8:26,3.39,17850.0,United Kingdom


#Step-2: Data Cleaning

In [3]:
#Fixing column Names
df.columns = df.columns.str.strip().str.lower().str.replace(" ","_")
print("Columns:", df.columns.tolist())

#Checking Null Values
print("\nNulls Before Cleaning:\n", df.isnull().sum())

#Dropping Rows where customer id is missing
df = df.dropna(subset = ["customerid"])

#Dropping rows where description is missing
df = df.dropna(subset = ["description"])

#Removing Cancelled Orders (Invoice starting with "C")
df = df[~df["invoiceno"].astype(str).str.startswith("C")]


#Removing orders with zero or negative quantity and price
df = df[df["quantity"]> 0 ]
df = df[df["unitprice"]> 0 ]

#Fixing Data Types
df["customerid"] = df["customerid"].astype(int)
df["invoicedate"] = pd.to_datetime(df["invoicedate"])

#Total Revenue Column
df["total_revenue"] = df["quantity"] * df["unitprice"]

#Year and Month
df["year"] = df["invoicedate"].dt.year
df["month"] = df["invoicedate"].dt.month

#Remove Duplicates
print("\nDuplicates Before Cleaning:\n", df.duplicated().sum())
print("\nDuplicates After Cleaning:\n", df.drop_duplicates().duplicated().sum())

#Confirm CLean
print("\nCleanShape:\n", df.shape)
print("\nNull Values After CLeaning:", df.isnull().sum())


Columns: ['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate', 'unitprice', 'customerid', 'country']

Nulls Before Cleaning:
 invoiceno          0
stockcode          0
description      839
quantity           1
invoicedate        1
unitprice          1
customerid     61412
country            1
dtype: int64

Duplicates Before Cleaning:
 1751

Duplicates After Cleaning:
 0

CleanShape:
 (148117, 11)

Null Values After CLeaning: invoiceno        0
stockcode        0
description      0
quantity         0
invoicedate      0
unitprice        0
customerid       0
country          0
total_revenue    0
year             0
month            0
dtype: int64


##Step-3: Load Clean data into SQL

In [4]:
#Create and in-memory SQLite database
conn = sqlite3.connect(":memory:")

#Load cleaned dataframe into SQL
df.to_sql("orders", conn, index = False, if_exists = "replace")

print("Data Loaded into SQL Successfully!")
print("Total Rows:", pd.read_sql("SELECT COUNT(*) as total FROM orders", conn).iloc[0,0])


Data Loaded into SQL Successfully!
Total Rows: 148117


##Step-4: Business Questions
Answering key business questions using SQL queries

###Q1: Which are the top 10 best selling products by total revenue?

In [5]:
q1 = pd.read_sql("""
      SELECT description, SUM(quantity) as total_quantity, ROUND(SUM(total_revenue),2) as total_revenue
      FROM orders
      GROUP BY description
      ORDER BY total_revenue DESC LIMIT 10
      """, conn)
print(q1)

                          description  total_quantity  total_revenue
0            REGENCY CAKESTAND 3 TIER          6949.0       79390.35
1      MEDIUM CERAMIC TOP STORAGE JAR         75055.0       78112.64
2  WHITE HANGING HEART T-LIGHT HOLDER         20810.0       56150.02
3                             POSTAGE          1251.0       35022.70
4                       PARTY BUNTING          7589.0       33687.35
5             JUMBO BAG RED RETROSPOT         18457.0       32963.86
6       ASSORTED COLOUR BIRD ORNAMENT         13691.0       22204.67
7                              Manual           534.0       20076.17
8                       CHILLI LIGHTS          4326.0       19472.06
9        VINTAGE UNION JACK MEMOBOARD          2663.0       17890.93


###Q2: Which are the Top 10 countries by total revenue?

In [6]:
q2 = pd.read_sql("""
      SELECT country, COUNT(DISTINCT invoiceno) as total_orders,ROUND(SUM(total_revenue),2) as total_revenue
      FROM orders
      GROUP BY country
      ORDER BY total_revenue DESC LIMIT 10
      """, conn)
print(q2)

          country  total_orders  total_revenue
0  United Kingdom          6820     2778328.79
1     Netherlands            38      118829.08
2         Germany           182       97433.91
3            EIRE            82       86071.64
4          France           149       73606.43
5       Australia            26       56379.98
6           Spain            32       24715.12
7           Japan            11       21434.73
8        Portugal            24       16260.41
9          Sweden            14       16163.33


###Q3: What is the Monthly revenue Trend ?

In [7]:
q3 = pd.read_sql("""
      SELECT year, month,
      ROUND(SUM(total_revenue),2) as total_revenue,
      COUNT(DISTINCT invoiceno) as total_orders
      FROM orders
      GROUP BY year, month
      ORDER BY year, month
      """, conn)
print(q3)

   year  month  total_revenue  total_orders
0  2010     12      572713.89          1400
1  2011      1      569445.04           987
2  2011      2      447137.35           997
3  2011      3      595500.76          1321
4  2011      4      469200.36          1149
5  2011      5      678594.56          1555
6  2011      6       64183.94           134


###Q4: Who are the top 10 customers by total spending ?


In [8]:
q4 = pd.read_sql("""
      SELECT customerid, COUNT(DISTINCT invoiceno) as total_orders,
      ROUND(SUM(total_revenue),2) as total_spent
      FROM orders
      GROUP BY customerid
      ORDER BY total_spent DESC LIMIT 10
      """, conn)
print(q4)

   customerid  total_orders  total_spent
0       14646            27    116135.92
1       12346             1     77183.60
2       18102            15     64642.11
3       17450            14     55760.26
4       12415             7     50883.90
5       15749             3     44534.30
6       14156            18     43907.23
7       14911            61     38799.78
8       16029            29     35776.26
9       17511            15     35750.40


###Q5: What is the average order value per country?

In [9]:
q5 = pd.read_sql("""
      SELECT country, ROUND(AVG(total_revenue), 2) as avg_order_value,
      COUNT(DISTINCT invoiceno) as total_orders
      FROM orders
      GROUP BY country
      ORDER BY avg_order_value
      DESC LIMIT 10
      """, conn)

print(q5)

       country  avg_order_value  total_orders
0  Netherlands           123.91            38
1    Australia           115.06            26
2       Sweden            97.96            14
3        Japan            95.69            11
4    Singapore            90.82             4
5      Denmark            79.17             4
6       Canada            61.34             2
7       Israel            48.69             1
8    Lithuania            47.46             4
9       Greece            43.04             3


###Q6: Using window function - Rank countries by revenue

In [10]:
q6 = pd.read_sql("""
      WITH country_revenue AS (
        SELECT country, ROUND(SUM(total_revenue), 2) AS total_revenue
        FROM orders
        GROUP BY country)
      SELECT country, total_revenue,
      RANK() OVER(ORDER BY total_revenue DESC ) AS revenue_rank
      FROM country_revenue
      LIMIT 10
      """, conn)
print(q6)

          country  total_revenue  revenue_rank
0  United Kingdom     2778328.79             1
1     Netherlands      118829.08             2
2         Germany       97433.91             3
3            EIRE       86071.64             4
4          France       73606.43             5
5       Australia       56379.98             6
6           Spain       24715.12             7
7           Japan       21434.73             8
8        Portugal       16260.41             9
9          Sweden       16163.33            10


## Key findings from the analysis

- Top selling product by revenue was "REGENCY CAKESTAND 3 TIER"
  indicating strong demand for home and kitchen products


- United Kingdom dominated revenue with ₹2,778,328 — accounting
  for approximately 81% of total global revenue


- May 2011 recorded the highest monthly revenue of ₹678,594
  with 1,555 orders. Note: dataset appears incomplete after
  June 2011 — full year trend may differ


- Top customer (ID: 14646) spent ₹116,135 - significantly higher
  than the average customer spend, indicating a high-value B2B buyer


- Netherlands led all countries with the highest average order
  value of ₹123.91 across 38 orders, followed by Australia
  (₹115.06) and Sweden (₹97.96) — suggesting these markets
  consist primarily of wholesale or B2B buyers placing
  high-value orders, unlike UK which has high volume
  but lower average order size
  

- Window function analysis confirmed UK, Netherlands and EIRE
  consistently rank in top 3 by revenue across all periods